# New wd=0.1 benchmark variant — exploratory follow-up
Prepared for execution, not independently scientifically certified. Original historical prose is background, not current evidence.
No new test access or adjacency edits. MB22 trains five models at wd=0.1 (not historical wd=0.01).
K32 and other training differences mean the single-vs-SEL comparison does not isolate training-code quality.
MB23 is a first-seed circuit analysis; its threshold verdicts are exploratory, not proof of a unique algorithm.


# MB22 — Making the Model Use Attention (Book 1 of 2): Two-Token Events and In-Context Player Identity

**Track B · Follow-up to MB14–MB21 · Identity × Context project**

| Item | Content |
|---|---|
| **Why** | In SEL-4L the attention heads play no part in the identity–context decision. Each event's token already carries a summary of the previous event, and the player has a global embedding. So everything the decision needs is in one token, and per-token MLPs do the work. This book trains models in which that information sits **at other positions**, so that attention has to fetch it. |
| **What changes (two changes combined)** | (1) **Two tokens per event.** A context token (what is known before the action) and an action token (what the event turned out to be). The previous-event fields are removed from the context token and live only in the previous event's action token. (2) **No global player identity.** Players get anonymous labels that are reshuffled in every window. The model can see which events in the window belong to the same player, but not who that player is. What it knows about the player must come from that player's earlier events in the window. |
| **Variants** | `single`: SEL-4L's input design trained with this code (control for the training code). `two_token_id`: change (1) only. `two_token_anon`: changes (1) + (2), **the main model**. |
| **Book 2 (MB23)** | Behaviour, attention, component tests and drawn circuits for these models, compared with SEL-4L. |
| **Rules** | SEL-4L is not changed. Validation only: no test data is read. Architecture and training settings are SEL-4L's (4 layers, 2 heads, d_model 128, d_ff 512, AdamW lr 3e-4, weight decay 0.1 (new benchmark variant), batch 256 windows, 5% warm-up, at most 30 epochs, patience 5). The plan is written and fingerprinted before training. |
| **Outputs** | `config/MB22_plan.json`, `mi_analysis/src/twotoken_followup.py`, `artifacts/twotoken/<variant>_s<seed>/`, `results/MB22_*` |

In [1]:
import os
import sys
import json
import time
import importlib
import importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")


def find_root(start):
    """`start` (or one of its ancestors) holding both project folders directly, else None."""
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "model_training_evaluation").is_dir() and (candidate / "mi_analysis").is_dir():
            return candidate
    return None


def search_root(bases, max_depth=6):
    """Breadth-first search under each base (e.g. a mounted Drive) for the project root."""
    for base in bases:
        base = Path(base)
        if not base.is_dir():
            continue
        found = find_root(base)
        if found is not None:
            return found
        frontier = [(base, 0)]
        while frontier:
            folder, depth = frontier.pop(0)
            if depth >= max_depth:
                continue
            try:
                children = [c for c in folder.iterdir() if c.is_dir() and not c.name.startswith(".")]
            except PermissionError:
                continue
            for child in children:
                if child.name == "model_training_evaluation" and (child.parent / "mi_analysis").is_dir():
                    return child.parent
                frontier.append((child, depth + 1))
    return None


if "SOCCER_MI_ROOT" in os.environ:
    ROOT = Path(os.environ["SOCCER_MI_ROOT"])
elif IN_COLAB:
    ROOT = search_root(["/content", "/content/drive/MyDrive", "/content/drive/My Drive"])
    if ROOT is None:
        raise RuntimeError(
            "BLOCKED: could not find a folder holding both model_training_evaluation/ and mi_analysis/ "
            "under /content or the mounted Drive. Upload or sync the project there, or set "
            "SOCCER_MI_ROOT to its exact path in a cell ABOVE this one, for example: "
            "import os; os.environ['SOCCER_MI_ROOT'] = '/content/drive/MyDrive/.../soccer_mi'")
else:
    ROOT = find_root(Path.cwd())
    if ROOT is None:
        raise RuntimeError(f"BLOCKED: no ancestor of {Path.cwd()} holds both model_training_evaluation/ "
                           "and mi_analysis/. Run this notebook from inside the project, or set SOCCER_MI_ROOT.")
TRACK_A_DIR = ROOT / "model_training_evaluation"
MI_DIR = Path(os.environ.get("MI_ANALYSIS_DIR", ROOT / "mi_analysis")).resolve()
assert (TRACK_A_DIR / "src" / "data.py").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'src' / 'data.py'} not found -- ROOT resolved to {ROOT}, which is wrong. "
    "Set SOCCER_MI_ROOT explicitly.")
assert (TRACK_A_DIR / "artifacts" / "events_prepared.parquet").exists(), (
    f"BLOCKED: {TRACK_A_DIR / 'artifacts' / 'events_prepared.parquet'} not found -- ROOT is wrong.")
RUN_MODE = os.environ.get("MI_RUN_MODE", "full")
assert RUN_MODE in ("full", "smoke")

for folder in [MI_DIR, MI_DIR / "src", MI_DIR / "notebooks", MI_DIR / "tests"]:
    folder.mkdir(parents=True, exist_ok=True)
(MI_DIR / "src" / "__init__.py").touch()
os.chdir(MI_DIR)

sys.path[:] = [entry for entry in sys.path if entry not in ("", ".", str(MI_DIR))]
if str(TRACK_A_DIR) not in sys.path:
    sys.path.insert(0, str(TRACK_A_DIR))
spec = importlib.util.spec_from_file_location("mi_src", MI_DIR / "src" / "__init__.py",
                                              submodule_search_locations=[str(MI_DIR / "src")])
mi_src = importlib.util.module_from_spec(spec)
sys.modules["mi_src"] = mi_src
spec.loader.exec_module(mi_src)

print("Colab:", IN_COLAB, "| RUN_MODE:", RUN_MODE)
print("ROOT:", ROOT)
print("Track A (read only):", TRACK_A_DIR, "| exists:", TRACK_A_DIR.exists())
print("Track B:", MI_DIR)

Colab: False | RUN_MODE: full
ROOT: /content/soccer_mi
Track A (read only): /content/soccer_mi/model_training_evaluation | exists: True
Track B: /content/soccer_mi/.mi_analysis_rerun


In [2]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import src.data as data
import src.metrics as metrics
import src.training as training
import mi_src.paths as paths
import mi_src.fingerprint as fingerprint
import mi_src.mi_setup as mi_setup
import mi_src.hooks as hooks
import mi_src.interventions as interventions
import mi_src.mi_metrics as mi_metrics
import mi_src.patching as patching
import mi_src.circuits as circuits
for module in [data, metrics, training, paths, fingerprint, mi_setup, hooks, interventions, mi_metrics, patching, circuits]:
    importlib.reload(module)

if getattr(paths, "SCHEMA_VERSION", 1) < 2:
    raise RuntimeError("BLOCKED: mi_analysis/src/paths.py is older than this notebook expects. "
                       "Re-run MB01 section 0 (its three %%writefile cells), then restart the kernel.")
paths.check_modules(**{name: module for name, module in
                       [("paths", paths), ("fingerprint", fingerprint), ("mi_setup", mi_setup),
                        ("patching", globals().get("patching")), ("circuits", globals().get("circuits"))]
                       if module is not None})

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

OUT = paths.output_dir(MI_DIR, RUN_MODE)
SETTINGS = paths.RUN_SETTINGS[RUN_MODE]
TRACK = paths.track_a_paths(TRACK_A_DIR)
MANIFEST = OUT / "manifests" / "sha_manifest.json"
REGISTRY = OUT / "manifests" / "experiment_registry.jsonl"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if RUN_MODE == "full" and DEVICE != "cuda":
    print("Note: no GPU found. The full run works on CPU but is much slower.")
training.configure_determinism()
training.set_seed(0)
ACTIONS = list(data.ACTION_NAMES)
print("Device:", DEVICE, "| outputs:", OUT)
print("Settings:", SETTINGS)
import torch.nn.functional as TF

Device: cuda | outputs: /content/soccer_mi/.mi_analysis_rerun
Settings: {'reference_epochs': 25, 'batch_windows': 256, 'n_bootstrap': 2000, 'n_draws': 5, 'n_permutations': 1000, 'n_random_circuits': 100, 'probe_train_events': 20000, 'max_events': 100000, 'probe_seeds': 3, 'scrub_draws': 5}


In [3]:
verified_at_start = fingerprint.verify_files(MANIFEST)
manifest_before = {name: entry["sha256"] for name, entry in fingerprint.load_manifest(MANIFEST).items()}
prereg = mi_setup.load_json(OUT / "config" / "preregistration.json")
mi_config = mi_setup.load_json(OUT / "config" / "mi_config.json")
IDENTITY = mi_config["identity_variable"]
assert IDENTITY == "player", "BLOCKED: MB14-MB16 follow up MB04-MB12, which studied the player variable."
PRIMARY_SEED = int(prereg["model"]["primary_seed"])


def run_dir(seed):
    """Run directory of one SEL-4L seed, re-rooted onto this checkout."""
    return paths.rebase_run_dir(prereg["model"]["run_dirs"][str(seed)], TRACK["root"])


SEEDS = [int(seed) for seed in prereg["model"]["run_dirs"]]
N_BOOT = SETTINGS["n_bootstrap"]
BATCH = SETTINGS["batch_windows"]
N_DRAWS = SETTINGS["n_draws"]

events = pd.read_parquet(TRACK["events"])
bundle = mi_setup.build_bundle(events, TRACK, DEVICE)
val_view = bundle["views"]["val"]
# No test view is built or read anywhere in MB14-MB16 (mi_setup.unlock_test_view is never called).
targets = bundle["targets"]
POP = OUT / "artifacts" / "populations"
val_conflicts = pd.read_parquet(POP / "val_conflicts.parquet")
controls = pd.read_parquet(POP / "matched_controls.parquet")
partners = pd.read_parquet(POP / "swap_partners.parquet")
donors = pd.read_parquet(POP / "context_donors.parquet")
partner_of = dict(zip(partners["player"], partners["partner"]))
reference = torch.load(POP / "reference_means.pt", map_location="cpu", weights_only=False)
MEANS = {site: value.to(DEVICE) for site, value in reference["means"].items()}
REFERENCE_WINDOWS = np.asarray(reference["windows"])
MODEL = mi_setup.load_model(run_dir(PRIMARY_SEED), bundle)
N_LAYERS = len(MODEL.blocks)
N_HEADS = MODEL.blocks[0].attention.n_heads
ALL = circuits.all_components(N_LAYERS, N_HEADS)
FEATURES = list(bundle["encoded"].tensors.keys())
IDENTITY_FEATURES = [name for name in ["player", "team", "role", "competition"] if name in FEATURES]
CONTEXT_FEATURES = [name for name in FEATURES if name not in IDENTITY_FEATURES]

p_player = val_conflicts[val_conflicts.population == "P-player"].reset_index(drop=True)
p_player = p_player.merge(donors[["event_row", "donor_window"]], on="event_row", how="left")
discovery = p_player[p_player.half == "discovery"].reset_index(drop=True)
estimation = p_player[p_player.half == "estimation"].reset_index(drop=True)
index0 = val_conflicts[val_conflicts.population == "index-0"].reset_index(drop=True)
print(f"verified files: {len(verified_at_start)} | primary seed {PRIMARY_SEED} | seeds {SEEDS}")
print(f"P-player rows: {len(p_player):,} (discovery {len(discovery):,}, estimation {len(estimation):,}) | "
      f"index-0: {len(index0):,} | matched controls: {len(controls):,}")
print(f"components: {ALL}")
print(f"identity inputs: {IDENTITY_FEATURES} | context inputs ({len(CONTEXT_FEATURES)}): {CONTEXT_FEATURES}")

/content/soccer_mi/model_training_evaluation/src/batching.py:19: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  self.tensors[name] = torch.as_tensor(np.asarray(values), device=device)


verified files: 108 | primary seed 103 | seeds [100, 101, 102, 103, 104]
P-player rows: 1,744 (discovery 1,215, estimation 529) | index-0: 8,947 | matched controls: 252
components: ['L0H0', 'L0H1', 'L0MLP', 'L1H0', 'L1H1', 'L1MLP', 'L2H0', 'L2H1', 'L2MLP', 'L3H0', 'L3H1', 'L3MLP']
identity inputs: ['player', 'team', 'role', 'competition'] | context inputs (15): ['zone', 'period', 'minute_bin', 'score_diff', 'play_pattern', 'poss_len', 'possession_flag', 'prev_action', 'prev_outcome', 'prev_end_zone', 'same_player_as_prev', 'x_norm', 'y_norm', 'dt', 't_poss']


## 1. Terms

| Term | Meaning here |
|---|---|
| **Context token c_t** | Event t before it happens: team, role, competition, zone and coordinates, period and minute, score, possession state, time gap, and the player (global embedding or anonymous label). |
| **Action token a_t** | What event t turned out to be: its action, outcome and end zone, plus the player and team. These are the values SEL-4L puts into event t+1's previous-event fields, so the information is the same, only one position earlier. |
| **Order and mask** | c_1, a_1, c_2, a_2, …; causal attention. The action of event t is predicted at c_t, which cannot see a_t. |
| **Anonymous label (slot)** | In every window, each player gets a random label from 1 to n_slots; labels are reshuffled every time a window is used in training, and fixed per window in evaluation. Events of the same player share a label within the window. |
| **In-context identity available** | The scored event's player has at least one earlier event in the window. Only then can the anonymous model know anything about the player. |
| **Window** | K = 32 events (64 tokens), stride 16. SEL-4L uses K = 16; the longer window gives the anonymous model more of each player's earlier events. Section 3 reports how often the player appears earlier at K = 16 and K = 32. |

## 2. The plan (written before training)

In [4]:
TT_EPOCHS = int(os.environ.get("TT_MAX_EPOCHS", "30"))
TT_SEEDS = [int(s) for s in os.environ.get("TT_SEEDS", "0,1,2").split(",")]
TT_LAYERS, TT_HEADS = int(os.environ.get("TT_LAYERS", "4")), int(os.environ.get("TT_HEADS", "2"))   # SEL-4L: 4 and 2; change only for code tests

In [5]:
PLAN = {
    "status": "follow-up after MB14-MB21; exploratory; new models next to SEL-4L, which is not changed",
    "variants": {"single": "SEL-4L inputs, one token per event (control for the training code)",
                 "two_token_id": "two tokens per event; history fields only in the previous action token; global player embedding",
                 "two_token_anon": "two tokens per event; history fields only in the previous action token; anonymous per-window player labels (main)"},
    "seeds": {"two_token_anon": TT_SEEDS, "two_token_id": TT_SEEDS[:1], "single": TT_SEEDS[:1]},
    "architecture": {"n_layers": TT_LAYERS, "n_heads": TT_HEADS, "d_model": 128, "d_ff": 512, "pre_layernorm": True, "dropout": 0, "n_events": 32},
    "training": {"optimiser": "AdamW", "learning_rate": 3e-4, "weight_decay": 0.1, "batch_windows": 256, "warmup_fraction": 0.05,
                 "schedule": "linear warm-up then cosine decay", "max_epochs": TT_EPOCHS, "patience": 5,
                 "selection": "epoch with the lowest validation NLL on scored validation events"},
    "evaluation": "validation only; each validation event scored once, in the window where it has the most preceding events",
    "reading_rules": {"same_information": "two_token_id is called equivalent to single if its validation NLL is within 0.01 nats of single's",
                      "anonymous_cost": "the cost of anonymous identity = NLL(two_token_anon) - NLL(two_token_id), reported overall and where in-context identity is available or not",
                      "training_code": "single is compared with SEL-4L on the same events; a gap > 0.02 nats means this training code is weaker than Track A's and every comparison is made within this book"}
}
plan_path = OUT / "config" / "MB22_plan.json"
if plan_path.exists():
    stored = mi_setup.load_json(plan_path)
    changed = {key for key in PLAN if key != "written_at" and stored.get(key) != PLAN[key]}
    if changed:
        raise RuntimeError(f"BLOCKED: the plan in {plan_path.name} differs from this notebook in {sorted(changed)}. "
                           "A plan is written once, before results; delete the file only if no result was read.")
    PLAN = stored
    print("plan already written at", stored["written_at"], "- unchanged")
else:
    PLAN["written_at"] = fingerprint.now()
    mi_setup.save_json(plan_path, PLAN)
    print("plan written:", plan_path.name)
fingerprint.record_files(MANIFEST, {"config/MB22_plan.json": plan_path})
print(json.dumps({key: value for key, value in PLAN.items() if key != "written_at"}, indent=1))

plan already written at 2026-10-08T05:30:25.145193+00:00 - unchanged
{
 "status": "follow-up after MB14-MB21; exploratory; new models next to SEL-4L, which is not changed",
 "variants": {
  "single": "SEL-4L inputs, one token per event (control for the training code)",
  "two_token_id": "two tokens per event; history fields only in the previous action token; global player embedding",
  "two_token_anon": "two tokens per event; history fields only in the previous action token; anonymous per-window player labels (main)"
 },
 "seeds": {
  "two_token_anon": [
   0,
   1,
   2
  ],
  "two_token_id": [
   0
  ],
  "single": [
   0
  ]
 },
 "architecture": {
  "n_layers": 4,
  "n_heads": 2,
  "d_model": 128,
  "d_ff": 512,
  "pre_layernorm": true,
  "dropout": 0,
  "n_events": 32
 },
 "training": {
  "optimiser": "AdamW",
  "learning_rate": 0.0003,
  "weight_decay": 0.1,
  "batch_windows": 256,
  "warmup_fraction": 0.05,
  "schedule": "linear warm-up then cosine decay",
  "max_epochs": 30,
  "

## 3. The model module, its checks, and the data

In [6]:
MODULE_SOURCE = r'''"""
mi_analysis/src/twotoken_followup.py — the two-token event Transformer with in-context player identity (MB22/MB23).

Written by MB22. Three variants share one code path:

  single          one token per event, every SEL-4L input (identity and history fields).
                  The control: SEL-4L's design, trained with this code.
  two_token_id    two tokens per event; the history fields are removed from the context token and
                  carried by the previous event's action token; the player keeps a global embedding.
  two_token_anon  as two_token_id, but the player has NO global embedding. Each window gives its players
                  random anonymous labels ("slots"), so the model can tell which events belong to the same
                  player, but not which player it is. Identity can only come from that player's earlier
                  events in the window, through attention. This is the main model.

Sequence layout of the two-token variants (K events -> 2K tokens):
  c_1, a_1, c_2, a_2, ..., c_K, a_K
  c_t (context token): pre-action fields of event t (team, role, competition, space, time, state) + player.
  a_t (action token):  what event t turned out to be: its action, outcome and end zone + player + team.
                       These are read from event t+1's previous-event fields, i.e. exactly the information
                       SEL-4L gives to token t+1. The prediction of event t's action is made at c_t, which
                       (causal mask) cannot see a_t.

The instrumented forward(model, inputs, replace, capture) mirrors mi_src.hooks.forward, with the same site
names, so the MI code of MB06-MB19 carries over.
"""
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.model import FACTOR_GROUPS, CONTINUOUS_GROUPS

HISTORY_FEATURES = ["prev_action", "prev_outcome", "prev_end_zone", "same_player_as_prev"]
ACTION_FIELDS = ["prev_action", "prev_outcome", "prev_end_zone"]        # read from event t+1
VARIANTS = ["single", "two_token_id", "two_token_anon"]
CONTEXT_GROUPS = ["team", "role", "competition", "space", "time", "state"]


class Config:
    def __init__(self, variant, sizes, n_events=32, n_layers=4, n_heads=2, d_model=128, d_ff=512, n_slots=None):
        assert variant in VARIANTS, variant
        self.variant, self.sizes = variant, dict(sizes)
        self.n_events, self.n_layers, self.n_heads, self.d_model, self.d_ff = n_events, n_layers, n_heads, d_model, d_ff
        self.n_slots = n_slots if n_slots is not None else n_events + 8
        self.two_token = variant != "single"
        self.n_tokens = 2 * n_events if self.two_token else n_events

    def to_dict(self):
        return {k: getattr(self, k) for k in ["variant", "sizes", "n_events", "n_layers", "n_heads", "d_model", "d_ff", "n_slots"]}

    @staticmethod
    def from_dict(d):
        return Config(d["variant"], d["sizes"], d["n_events"], d["n_layers"], d["n_heads"], d["d_model"], d["d_ff"], d["n_slots"])


class Attention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_heads, self.d_head = c.n_heads, c.d_model // c.n_heads
        self.W_Q, self.W_K, self.W_V, self.W_O = (nn.Linear(c.d_model, c.d_model) for _ in range(4))
        self.register_buffer("causal_mask", torch.tril(torch.ones(c.n_tokens, c.n_tokens, dtype=torch.bool)))

    def split_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.d_head).transpose(1, 2)


class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln_attention, self.attention = nn.LayerNorm(c.d_model), Attention(c)
        self.ln_mlp = nn.LayerNorm(c.d_model)
        self.mlp = nn.Module()
        self.mlp.input_layer, self.mlp.output_layer = nn.Linear(c.d_model, c.d_ff), nn.Linear(c.d_ff, c.d_model)


class TwoTokenModel(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.config = c
        D = c.d_model
        self.embeddings = nn.ModuleDict()
        groups = (["player"] if c.variant != "two_token_anon" else []) + CONTEXT_GROUPS + (["history"] if c.variant == "single" else [])
        self.context_groups = groups
        for group in groups:
            for feature in FACTOR_GROUPS[group]:
                self.embeddings[feature] = nn.Embedding(c.sizes[feature], D)
        self.continuous = nn.ModuleDict({g: nn.Linear(len(v), D, bias=False) for g, v in CONTINUOUS_GROUPS.items() if g in groups})
        if c.variant == "two_token_anon":
            self.slot_embedding = nn.Embedding(c.n_slots, D)
        if c.two_token:
            self.action_embeddings = nn.ModuleDict({f: nn.Embedding(c.sizes[f], D) for f in ACTION_FIELDS})
            self.type_embedding = nn.Embedding(2, D)
        self.position_embedding = nn.Embedding(c.n_events, D)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layers)])
        self.ln_final = nn.LayerNorm(D)
        self.unembed = nn.Linear(D, 15)
        with torch.no_grad():
            if "player" in self.embeddings:
                self.embeddings["player"].weight[0].zero_()
            if c.variant == "two_token_anon":
                self.slot_embedding.weight[0].zero_()

    def context_positions(self):
        c = self.config
        return torch.arange(c.n_events) * (2 if c.two_token else 1)


def _group_vector(model, inputs, group):
    vector = None
    for feature in FACTOR_GROUPS[group]:
        v = model.embeddings[feature](inputs[feature])
        vector = v if vector is None else vector + v
    if group in model.continuous:
        values = torch.stack([inputs[name] for name in CONTINUOUS_GROUPS[group]], dim=-1)
        vector = vector + model.continuous[group](values)
    return vector


def embed_factors(model, inputs):
    """{site name: [B, T, D]} — one vector per input group, zero at positions where the group is absent."""
    c = model.config
    B, K = inputs["zone"].shape
    D = c.d_model
    out = {}

    def place(event_vectors, kind):
        if not c.two_token:
            return event_vectors
        full = event_vectors.new_zeros(B, 2 * K, D)
        full[:, (0 if kind == "context" else 1)::2] = event_vectors
        return full

    identity_name = "slot" if c.variant == "two_token_anon" else "player"
    for group in model.context_groups:
        name = identity_name if group == "player" else group
        out[f"factor_{name}"] = place(_group_vector(model, inputs, group), "context")
    if c.variant == "two_token_anon":
        slot_vec = model.slot_embedding(inputs["slot"])
        out["factor_slot"] = place(slot_vec, "context")
    if c.two_token:
        act = None
        for f in ACTION_FIELDS:
            v = model.action_embeddings[f](inputs[f"act_{f}"])
            act = v if act is None else act + v
        out["factor_act_action"] = place(act, "action")
        ident = model.slot_embedding(inputs["slot"]) if c.variant == "two_token_anon" else model.embeddings["player"](inputs["player"])
        out[f"factor_act_{identity_name}"] = place(ident, "action")
        out["factor_act_team"] = place(model.embeddings["team"](inputs["team"]), "action")
        types = torch.arange(2 * K, device=inputs["zone"].device) % 2
        out["factor_type"] = model.type_embedding(types).unsqueeze(0).expand(B, -1, -1)
        positions = torch.arange(2 * K, device=inputs["zone"].device) // 2
    else:
        positions = torch.arange(K, device=inputs["zone"].device)
    out["factor_position"] = model.position_embedding(positions).unsqueeze(0).expand(B, -1, -1)
    return out


def forward(model, inputs, replace=None, capture=None):
    """(logits [B, T, 15], cache). Same site names as mi_src.hooks.forward."""
    replace = replace or {}
    cache = {}

    def site(name, value):
        if name in replace:
            value = replace[name](value)
        if capture is not None and (capture == "all" or name in capture):
            cache[name] = value.detach()
        return value

    residual = None
    for name, vector in embed_factors(model, inputs).items():
        vector = site(name, vector)
        residual = vector if residual is None else residual + vector
    residual = site("resid_pre_0", residual)
    B, T, _ = residual.shape
    for layer, block in enumerate(model.blocks):
        if layer > 0:
            residual = site(f"resid_pre_{layer}", residual)
        attention = block.attention
        shared = block.ln_attention(residual)
        q, k, v = (attention.split_heads(p(shared)) for p in (attention.W_Q, attention.W_K, attention.W_V))
        for head in range(attention.n_heads):
            for kind, projection, tensor in [("q", attention.W_Q, q), ("k", attention.W_K, k), ("v", attention.W_V, v)]:
                name = f"attn_in_{kind}_{layer}.{head}"
                if name in replace or (capture is not None and capture != "all" and name in capture):
                    receiver_input = site(name, residual)
                    if name in replace:
                        tensor[:, head] = attention.split_heads(projection(block.ln_attention(receiver_input)))[:, head]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(attention.d_head)
        scores = scores.masked_fill(~attention.causal_mask[:T, :T], float("-inf"))
        scores = site(f"attn_scores_{layer}", scores)
        pattern = site(f"attn_pattern_{layer}", torch.softmax(scores, dim=-1))
        z = site(f"head_z_{layer}", torch.matmul(pattern, v).transpose(1, 2))
        attn_out = site(f"attn_out_{layer}", attention.W_O(z.reshape(B, T, -1)))
        residual = site(f"resid_mid_{layer}", residual + attn_out)
        mlp_in = site(f"mlp_in_{layer}", residual)
        hidden = site(f"mlp_hidden_{layer}", F.gelu(block.mlp.input_layer(block.ln_mlp(mlp_in))))
        mlp_out = site(f"mlp_out_{layer}", block.mlp.output_layer(hidden))
        residual = site(f"resid_post_{layer}", residual + mlp_out)
    final = site("final_resid", residual)
    normalised = site("final_normalised", model.ln_final(final))
    return model.unembed(normalised), cache


# ----------------------------------------------------------------------------------------------- data

class EventData:
    """Per-event arrays (model's own T300 encoding) and the windows of each split.

    events: events_prepared (row order = time order within a match); encoded: {feature: tensor over all events}.
    """

    def __init__(self, events, encoded, targets, n_events, stride=None):
        self.n_events = n_events
        self.stride = stride or n_events // 2
        self.features = {name: tensor.detach().cpu().numpy() for name, tensor in encoded.items()}
        self.targets = np.asarray(targets).astype(np.int64)
        match = events["match_id"].to_numpy()
        n = len(events)
        same_next = np.zeros(n, dtype=bool)
        same_next[:-1] = match[1:] == match[:-1]
        for f in ACTION_FIELDS:
            nxt = np.zeros(n, dtype=self.features[f].dtype)
            nxt[:-1] = self.features[f][1:]
            self.features[f"act_{f}"] = np.where(same_next, nxt, 0)
        self.match = match
        self.split = events["split"].to_numpy()
        self.windows, self.scored = {}, {}
        for split in ("train", "val"):
            self._build(split)

    def _build(self, split):
        K, S = self.n_events, self.stride
        rows_all = np.where(self.split == split)[0]
        windows, scored_rows, scored_window, scored_pos = [], [], [], []
        for m in np.unique(self.match[rows_all]):
            rows = rows_all[self.match[rows_all] == m]
            if len(rows) < K:
                continue
            starts = list(range(0, len(rows) - K + 1, S))
            if starts[-1] != len(rows) - K:
                starts.append(len(rows) - K)
            first = len(windows)
            for s in starts:
                windows.append(rows[s:s + K])
            starts = np.array(starts)
            for i, r in enumerate(rows):
                ok = np.where((starts <= i) & (i < starts + K))[0]
                best = ok[np.argmax(i - starts[ok])]
                scored_rows.append(r); scored_window.append(first + best); scored_pos.append(i - starts[best])
        self.windows[split] = np.array(windows)
        self.scored[split] = {"event_row": np.array(scored_rows), "window": np.array(scored_window), "position": np.array(scored_pos)}

    def batch(self, split, window_ids, device, slot_rng=None, n_slots=None):
        rows = self.windows[split][np.asarray(window_ids)]
        out = {}
        for name, values in self.features.items():
            dtype = torch.float32 if values.dtype.kind == "f" else torch.long
            out[name] = torch.as_tensor(values[rows], dtype=dtype, device=device)
        players = self.features["player"][rows]
        slots = np.zeros_like(players)
        rng = slot_rng if slot_rng is not None else np.random.default_rng(0)
        for b in range(len(rows)):
            unique = [p for p in dict.fromkeys(players[b].tolist()) if p > 0]
            labels = rng.permutation(np.arange(1, n_slots))[:len(unique)]
            lookup = dict(zip(unique, labels))
            slots[b] = [lookup.get(p, 0) for p in players[b]]
        out["slot"] = torch.as_tensor(slots, dtype=torch.long, device=device)
        return out, torch.as_tensor(self.targets[rows], dtype=torch.long, device=device)

    def eval_slot_rng(self, split, window_id):
        """Deterministic anonymous labels for evaluation: one fixed permutation per window."""
        return np.random.default_rng(1_000_003 * {"train": 1, "val": 2, "test": 3}.get(str(split), 0) + int(window_id))


def eval_batch(data, split, window_ids, device, n_slots):
    """Evaluation inputs with fixed slots per window (same labels every time a window is used)."""
    parts, targets = [], []
    for w in np.asarray(window_ids):
        inputs, t = data.batch(split, [w], device, slot_rng=data.eval_slot_rng(split, w), n_slots=n_slots)
        parts.append(inputs); targets.append(t)
    return {k: torch.cat([p[k] for p in parts]) for k in parts[0]}, torch.cat(targets)


def context_index(model, event_positions):
    """Token index of the context token of each event position."""
    return event_positions * (2 if model.config.two_token else 1)


def logits_at_events(logits, model):
    """[B, K, 15] logits read at every event's context token."""
    return logits[:, ::2] if model.config.two_token else logits
'''

module_path = MI_DIR / "src" / "twotoken_followup.py"
module_path.write_text(MODULE_SOURCE)
import mi_src.twotoken_followup as tt
importlib.reload(tt)
fingerprint.record_files(MANIFEST, {"src/twotoken_followup.py": module_path})
print("written and imported", module_path)

written and imported /content/soccer_mi/.mi_analysis_rerun/src/twotoken_followup.py


In [7]:
SIZES = {name: int(tensor.max().item()) + 1 for name, tensor in bundle["encoded"].tensors.items() if not tensor.is_floating_point()}
TT_DATA = tt.EventData(events, bundle["encoded"].tensors, bundle["targets"], n_events=32)
for split, windows in TT_DATA.windows.items():
    print(f"{split}: {len(windows):,} windows of {TT_DATA.n_events} events | scored events {len(TT_DATA.scored[split]['event_row']):,}")
assert "test" not in TT_DATA.windows


def appears_earlier(data, split):
    scored = data.scored[split]
    rows = data.windows[split][scored["window"]]
    players = data.features["player"][rows]
    own = players[np.arange(len(rows)), scored["position"]]
    earlier = np.array([own[i] > 0 and (players[i, :scored["position"][i]] == own[i]).any() for i in range(len(rows))])
    return earlier


coverage = {}
for K in [16, 32]:
    data_k = TT_DATA if K == 32 else tt.EventData(events, bundle["encoded"].tensors, bundle["targets"], n_events=K)
    coverage[K] = appears_earlier(data_k, "val")
    rows = data_k.scored["val"]["event_row"]
    is_pplayer = np.isin(rows, p_player["event_row"].to_numpy())
    print(f"K = {K}: scored validation events whose player appears earlier in the window: {coverage[K].mean():.3f} "
          f"(P-player conflicts: {coverage[K][is_pplayer].mean():.3f})")

train: 32,721 windows of 32 events | scored events 525,893
val: 4,229 windows of 32 events | scored events 68,003


K = 16: scored validation events whose player appears earlier in the window: 0.121 (P-player conflicts: 0.513)
K = 32: scored validation events whose player appears earlier in the window: 0.150 (P-player conflicts: 0.657)


**Built-in checks.**

1. Each action token carries exactly event t+1's previous-event fields.
2. **No leak:** changing a_t (or anything after c_t) must leave the prediction at c_t unchanged.
3. The model's prediction at c_t depends on a_{t−1} (the information is reachable).

In [8]:
rows_check = TT_DATA.windows["val"][0]
for f in tt.ACTION_FIELDS:
    same_match = TT_DATA.match[rows_check[:-1]] == TT_DATA.match[rows_check[1:]]
    assert np.array_equal(TT_DATA.features[f"act_{f}"][rows_check[:-1]][same_match], TT_DATA.features[f][rows_check[1:]][same_match]), f
torch.manual_seed(0)
probe = tt.TwoTokenModel(tt.Config("two_token_anon", SIZES)).to(DEVICE).eval()
inputs_check, _ = tt.eval_batch(TT_DATA, "val", [0, 1], DEVICE, probe.config.n_slots)
with torch.no_grad():
    base = tt.logits_at_events(tt.forward(probe, inputs_check)[0], probe)
    t = 10
    changed = {k: v.clone() for k, v in inputs_check.items()}
    for f in tt.ACTION_FIELDS:
        changed[f"act_{f}"][:, t] = (changed[f"act_{f}"][:, t] + 1) % SIZES[f]
    after = tt.logits_at_events(tt.forward(probe, changed)[0], probe)
assert torch.allclose(base[:, :t + 1], after[:, :t + 1]), "BLOCKED: the prediction at c_t depends on a_t or later (leak)"
assert not torch.allclose(base[:, t + 1], after[:, t + 1]), "BLOCKED: a_t does not reach c_(t+1)"
print("checks passed: action tokens = next event's history fields; no leak; a_t reaches c_(t+1)")

checks passed: action tokens = next event's history fields; no leak; a_t reaches c_(t+1)


## 4. Training

Training windows use every context token's prediction. Anonymous labels are redrawn for every batch. The best epoch (lowest validation NLL) is kept. A finished run is not retrained: its checkpoint is reloaded.

In [9]:
TT_DIR = OUT / "artifacts" / "twotoken"
TT_DIR.mkdir(parents=True, exist_ok=True)
VAL_SCORED = TT_DATA.scored["val"]
VAL_WINDOWS = np.arange(len(TT_DATA.windows["val"]))
VAL_INPUTS = []
for start in range(0, len(VAL_WINDOWS), BATCH):
    inputs_v, targets_v = tt.eval_batch(TT_DATA, "val", VAL_WINDOWS[start:start + BATCH], "cpu", tt.Config("two_token_anon", SIZES).n_slots)
    VAL_INPUTS.append((inputs_v, targets_v))


@torch.no_grad()
def window_log_probs(model):
    """[n_val_windows, K, 15] log-probabilities at every context token (fixed evaluation labels)."""
    model.eval()
    out = []
    for inputs_v, _ in VAL_INPUTS:
        inputs_v = {k: v.to(DEVICE) for k, v in inputs_v.items()}
        out.append(torch.log_softmax(tt.logits_at_events(tt.forward(model, inputs_v)[0], model), -1).cpu())
    return torch.cat(out)


def scored_nll(log_probs):
    lp = log_probs[torch.as_tensor(VAL_SCORED["window"]), torch.as_tensor(VAL_SCORED["position"])]
    targets = torch.as_tensor(TT_DATA.targets[VAL_SCORED["event_row"]])
    return -lp[torch.arange(len(targets)), targets].numpy()


def train_variant(variant, seed):
    run_dir = TT_DIR / f"{variant}_s{seed}"
    if all((run_dir / name).exists() for name in ("best.pt", "config.json", "history.json")):
        config = tt.Config.from_dict(json.loads((run_dir / "config.json").read_text()))
        model = tt.TwoTokenModel(config).to(DEVICE)
        model.load_state_dict(torch.load(run_dir / "best.pt", map_location=DEVICE))
        fingerprint.record_files(MANIFEST, {f"artifacts/twotoken/{variant}_s{seed}/best.pt": run_dir / "best.pt"})
        print(f"{variant} seed {seed}: reloaded completed run")
        return model, json.loads((run_dir / "history.json").read_text())
    run_dir.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(seed); np.random.seed(seed)
    arch = PLAN["architecture"]
    config = tt.Config(variant, SIZES, n_events=arch["n_events"], n_layers=arch["n_layers"], n_heads=arch["n_heads"],
                       d_model=arch["d_model"], d_ff=arch["d_ff"])
    model = tt.TwoTokenModel(config).to(DEVICE)
    optimiser = torch.optim.AdamW(model.parameters(), lr=PLAN["training"]["learning_rate"], weight_decay=PLAN["training"]["weight_decay"])
    train_windows = np.arange(len(TT_DATA.windows["train"]))
    batch_size = PLAN["training"]["batch_windows"]
    steps_per_epoch = int(np.ceil(len(train_windows) / batch_size))
    total = steps_per_epoch * PLAN["training"]["max_epochs"]
    warmup = max(1, int(PLAN["training"]["warmup_fraction"] * total))
    schedule = torch.optim.lr_scheduler.LambdaLR(optimiser, lambda s: (s + 1) / warmup if s < warmup
                                                 else 0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, total - warmup))))
    rng = np.random.default_rng(seed)
    best, best_epoch, history, waited = np.inf, -1, [], 0
    start = time.time()
    for epoch in range(PLAN["training"]["max_epochs"]):
        model.train()
        order = rng.permutation(train_windows)
        losses = []
        for b in range(steps_per_epoch):
            inputs_b, targets_b = TT_DATA.batch("train", order[b * batch_size:(b + 1) * batch_size], DEVICE, slot_rng=rng, n_slots=config.n_slots)
            logits = tt.logits_at_events(tt.forward(model, inputs_b)[0], model)
            loss = TF.cross_entropy(logits.reshape(-1, 15), targets_b.reshape(-1))
            optimiser.zero_grad(); loss.backward(); optimiser.step(); schedule.step()
            losses.append(loss.item())
        val_nll = float(scored_nll(window_log_probs(model)).mean())
        history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), "val_nll": val_nll})
        if val_nll < best - 1e-5:
            best, best_epoch, waited = val_nll, epoch, 0
            torch.save(model.state_dict(), run_dir / "best.pt")
        else:
            waited += 1
        print(f"{variant} s{seed} epoch {epoch:2d}: train {np.mean(losses):.4f} | val {val_nll:.4f} | best {best:.4f} @ {best_epoch} | {time.time() - start:.0f} s")
        if waited >= PLAN["training"]["patience"]:
            break
    (run_dir / "config.json").write_text(json.dumps(config.to_dict()))
    (run_dir / "history.json").write_text(json.dumps(history))
    model.load_state_dict(torch.load(run_dir / "best.pt", map_location=DEVICE))
    fingerprint.record_files(MANIFEST, {f"artifacts/twotoken/{variant}_s{seed}/best.pt": run_dir / "best.pt"})
    return model, history


import math
MODELS_TT = {}
for variant, seeds in PLAN["seeds"].items():
    for seed in seeds:
        MODELS_TT[(variant, seed)] = train_variant(variant, seed)[0]

two_token_anon seed 0: reloaded completed run
two_token_anon seed 1: reloaded completed run
two_token_anon seed 2: reloaded completed run
two_token_id seed 0: reloaded completed run
single seed 0: reloaded completed run


## 5. Validation results

All rows use the same validation events (each scored once). SEL-4L is scored by its own Track A view on the same events. P-player conflicts are the MI population of MB04–MB21.

In [10]:
sel_table = circuits.scored_events(val_view, targets, bundle["match_ids"])
sel_table = sel_table[sel_table["event_row"].isin(VAL_SCORED["event_row"])].reset_index(drop=True)
sel_nll = pd.Series(circuits.event_nll(MODEL, bundle, val_view, sel_table, BATCH), index=sel_table["event_row"].to_numpy())
rows_val = VAL_SCORED["event_row"]
groups = {"all validation events": np.ones(len(rows_val), dtype=bool),
          "player appears earlier in the window": coverage[32],
          "player does not appear earlier": ~coverage[32],
          "P-player conflicts": np.isin(rows_val, p_player["event_row"].to_numpy())}
match_of = TT_DATA.match[rows_val]
result_rows = []
nll_by_run = {}
for (variant, seed), model in MODELS_TT.items():
    nll_by_run[(variant, seed)] = scored_nll(window_log_probs(model))
nll_by_run[("SEL-4L (Track A)", PRIMARY_SEED)] = sel_nll.reindex(rows_val).to_numpy()
for (variant, seed), nll in nll_by_run.items():
    for group, keep in groups.items():
        keep = keep & np.isfinite(nll)
        summary = mi_metrics.summarise(nll[keep], match_of[keep], N_BOOT)
        result_rows.append({"model": variant, "seed": seed, "events": group, "n": summary["n"], "NLL": summary["mean"],
                            "CI low": summary["ci_low"], "CI high": summary["ci_high"]})
results_tt = pd.DataFrame(result_rows)
results_tt.to_csv(OUT / "results" / "MB22_validation.csv", index=False)
print(results_tt.pivot_table(index=["model", "seed"], columns="events", values="NLL").round(4).to_string())

events                 P-player conflicts  all validation events  player appears earlier in the window  player does not appear earlier
model            seed                                                                                                                 
SEL-4L (Track A) 103               0.4310                 0.9038                                0.8888                          0.9064
single           0                 0.4745                 0.9480                                0.9245                          0.9521
two_token_anon   0                 0.5075                 0.9570                                0.9393                          0.9602
                 1                 0.5015                 0.9641                                0.9463                          0.9672
                 2                 0.5035                 0.9976                                0.9431                          1.0072
two_token_id     0                 0.5485              

In [11]:
def paired(a, b, group):
    keep = groups[group] & np.isfinite(nll_by_run[a]) & np.isfinite(nll_by_run[b])
    s = mi_metrics.summarise(nll_by_run[a][keep] - nll_by_run[b][keep], match_of[keep], N_BOOT)
    return {"comparison": f"{a[0]} s{a[1]} minus {b[0]} s{b[1]}", "events": group, "difference (nats)": s["mean"], "CI low": s["ci_low"], "CI high": s["ci_high"]}


first = TT_SEEDS[0]
comparisons = []
for group in groups:
    comparisons.append(paired(("single", first), ("SEL-4L (Track A)", PRIMARY_SEED), group))
    comparisons.append(paired(("two_token_id", first), ("single", first), group))
    comparisons.append(paired(("two_token_anon", first), ("two_token_id", first), group))
comparisons = pd.DataFrame(comparisons)
comparisons.to_csv(OUT / "results" / "MB22_comparisons.csv", index=False)
print(comparisons.round(4).to_string(index=False))
row = comparisons[(comparisons.comparison.str.startswith("two_token_id")) & (comparisons.events == "all validation events")].iloc[0]
print(f"\nsame information, different place (two_token_id vs single): {row['difference (nats)']:+.4f} nats "
      f"-> equivalent by the plan rule (|diff| <= 0.01): {abs(row['difference (nats)']) <= 0.01}")
row = comparisons[(comparisons.comparison.str.startswith("single")) & (comparisons.events == "all validation events")].iloc[0]
print(f"training code check (single vs SEL-4L): {row['difference (nats)']:+.4f} nats -> code weaker than Track A's (> 0.02): {row['difference (nats)'] > 0.02}")
anon = results_tt[results_tt.model == "two_token_anon"].pivot_table(index="seed", columns="events", values="NLL")
print("\ntwo_token_anon across seeds:\n" + anon.round(4).to_string() + f"\nseed SD (all events): {anon['all validation events'].std():.4f}")

                             comparison                               events  difference (nats)  CI low  CI high
  single s0 minus SEL-4L (Track A) s103                all validation events             0.0442  0.0368   0.0519
        two_token_id s0 minus single s0                all validation events             0.0279  0.0087   0.0533
two_token_anon s0 minus two_token_id s0                all validation events            -0.0189 -0.0311  -0.0090
  single s0 minus SEL-4L (Track A) s103 player appears earlier in the window             0.0357  0.0262   0.0435
        two_token_id s0 minus single s0 player appears earlier in the window             0.0345  0.0072   0.0749
two_token_anon s0 minus two_token_id s0 player appears earlier in the window            -0.0196 -0.0282  -0.0123
  single s0 minus SEL-4L (Track A) s103       player does not appear earlier             0.0457  0.0375   0.0545
        two_token_id s0 minus single s0       player does not appear earlier             0.0268 

**How to read it.**

- **`single` vs SEL-4L** checks this book's training code. If `single` is clearly worse, every comparison should be made within this book (`two_token_*` against `single`), not against SEL-4L.
- **`two_token_id` vs `single`** asks whether moving the previous-event fields into the previous action token loses anything. If they are equivalent, the two designs carry the same information, and any difference in mechanism found in MB23 comes from *where* the information sits.
- **`two_token_anon` vs `two_token_id`** is the price of having no global player identity. Where the player appears earlier in the window, the anonymous model can recover some of it in context. Where the player does not appear earlier, it cannot recover any.

In [12]:
summary = {"status": PLAN["status"], "coverage_K16": float(coverage[16].mean()), "coverage_K32": float(coverage[32].mean()),
           "validation": results_tt.to_dict(orient="records"), "comparisons": comparisons.to_dict(orient="records"),
           "runs": [f"{v}_s{s}" for v, s in MODELS_TT], "run_mode": RUN_MODE}
mi_setup.save_json(OUT / "results" / "MB22_summary.json", summary)
fingerprint.record_files(MANIFEST, {"results/MB22_summary.json": OUT / "results" / "MB22_summary.json"})
fingerprint.register(REGISTRY, {"notebook": "MB22", "experiment": "two-token and anonymous-identity variants trained", "run_mode": RUN_MODE})

In [13]:
verified_at_end = fingerprint.verify_files(MANIFEST)
manifest_after = {name: entry["sha256"] for name, entry in fingerprint.load_manifest(MANIFEST).items()}
changed = sorted(name for name in manifest_before if manifest_after.get(name) != manifest_before[name])
print(f"manifest entries verified at the end: {len(verified_at_end)}")
print("earlier entries whose fingerprint changed during this notebook:", changed if changed else "none")
assert not changed, "BLOCKED: an earlier Track B file changed while this notebook ran."
test_log = OUT / "logs" / "test_access_log.jsonl"
test_lines = len(test_log.read_text().strip().splitlines()) if test_log.exists() and test_log.read_text().strip() else 0
print(f"test accesses recorded so far: {test_lines} (MB12 recorded exactly one; this notebook adds none)")

manifest entries verified at the end: 109
earlier entries whose fingerprint changed during this notebook: none
test accesses recorded so far: 1 (MB12 recorded exactly one; this notebook adds none)


**Researcher interpretation** *(written by the researcher after reading the numbers above; the code never writes these words)*

- What was tested:
- What was observed:
- What it supports / contradicts:
- What it does not establish:
- Evidence label: